In [ ]:
# uberfigure
import dascore as dc
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker
from dascore.units import Hz
import numpy as np
import os

def matplotlib_wiggle(ax, data, times, scale, linewidth, alpha):
    """
    Plots a DAS patch as a wiggle plot using only Matplotlib.
    """

    # Quit if empty
    if data.size == 0: 
        print(f"matplotlib_wiggle has quit (empty data)")
        return

    # Quit if mismatch
    n_samples_from_coords = len(times) 
    if data.shape[0] != n_samples_from_coords:
            print(f"Data shape {data.shape} does not match time samples {n_samples_from_coords}")
            return
    
    n_samples, n_channels = data.shape

    try:
        # Scaling by 99th percentile extrema.  Old debugging method, but still works fine so it's kept.
        global_max = np.nanpercentile(np.abs(data), 99)
        if global_max == 0 or not np.isfinite(global_max):
            global_max = 1.0 
    except ValueError:
        global_max = 1.0
    
    # This draws each waveform (scaled-trace) around each channel index i
    for i in range(n_channels):
        trace = data[:, i]
        if np.all(np.isnan(trace)):
            continue
                
        scaled_trace = (trace / global_max) * scale
        scaled_trace = np.nan_to_num(scaled_trace)
        x_plot = times
        y_plot = i + scaled_trace
        ax.plot(x_plot, y_plot, color='black', linewidth=linewidth, alpha=alpha)
    
    ax.set_ylim(-1, n_channels)

def plot_zoomed_wiggle(patch, ax, title):
    """
    Processes and plots an UNDECIMATED DAS patch as a wiggle plot.
    """
    try:
        process1_patch = patch.dropna(dim="time", how="all")
        processed_patch = process1_patch.pass_filter(time=(1 * Hz, None))
        data = processed_patch.data
        times = processed_patch.coords.get_array('time')
        matplotlib_wiggle(ax, data, times, scale=2, linewidth=1, alpha=0.5)        
        ax.set_title(title, fontsize=10)
        ax.invert_yaxis()
        
    except Exception as e:
        print(f"Failed to plot wiggle for {title}: {e}")
        ax.set_title(f"Failed to plot: {title}")

def draw_zoom_box(ax_ref, ref_dist_coords, time_range, distance_range, label_text):
    """
    Draws a labeled rectangle on the reference axis (ax_ref).
    Converts distance (e.g., meters) to y-axis data coordinates (channel index).
    Takes a raw distance array.
    """
    y_start_index = np.argmin(np.abs(ref_dist_coords - distance_range[0]))
    y_end_index = np.argmin(np.abs(ref_dist_coords - distance_range[1]))
    y1 = min(y_start_index, y_end_index)
    y2 = max(y_start_index, y_end_index)
    rect_y = y1
    rect_height = y2 - y1
    rect_x_time = time_range[0]
    rect_width_time = time_range[1] - time_range[0]
    rect = patches.Rectangle((rect_x_time, rect_y), rect_width_time, rect_height,
                             linewidth=1.5, edgecolor='red', facecolor='none', linestyle='--')
    ax_ref.add_patch(rect)
    text_y_coord = rect_y + (rect_height / 2)
    text_x_coord = rect_x_time - (rect_width_time / 2)
    ax_ref.text(text_x_coord, text_y_coord, label_text, color='red', ha='center', va='center', 
                fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.9, pad=0.1, boxstyle='square'))

def set_meter_ticks(ax, dist_coords, tick_interval_m=1000):
    """
    Plots y-ticks and labels in meters based on the actual graph's dimension.
    Takes a raw distance array (dist_coords) corresponding to channel indices.
    """
    try:
        min_dist = np.min(dist_coords)
        max_dist = np.max(dist_coords)
        start_m = np.ceil(min_dist / tick_interval_m) * tick_interval_m
        end_m = np.floor(max_dist / tick_interval_m) * tick_interval_m
        if end_m < start_m:
            ax.set_ylabel("Distance (m)")
            return

        meter_labels = np.arange(start_m, end_m + tick_interval_m, tick_interval_m)
        tick_indices = []
        for meter_val in meter_labels:
            index = np.argmin(np.abs(dist_coords - meter_val))
            tick_indices.append(index)
            
        ax.set_yticks(tick_indices)
        ax.set_yticklabels([f"{int(m)}" for m in meter_labels])
        ax.set_ylabel("Distance (m)", fontsize=12)

    except Exception as e:
        print(f"Failed to set meter ticks: {e}")
        ax.set_ylabel("Channel Index") 

def combine_patches(kkfls_patch_in, terra_patch_in):
    """
    Combines KKFLS and TERRA patches for plotting on one graph,
    and returns three numpy arrays: (sorted_data, times, sorted_dist)
    """
    data_kkfls = kkfls_patch_in.data
    dist_kkfls = kkfls_patch_in.coords.get_array('distance') * -1
    data_terra = terra_patch_in.data
    dist_terra = terra_patch_in.coords.get_array('distance')
    
    times = kkfls_patch_in.coords.get_array('time')
    kkfls_zero_idx = np.where(dist_kkfls == 0)[0]
    terra_zero_idx = np.where(dist_terra == 0)[0]

    if kkfls_zero_idx.size > 0 and terra_zero_idx.size > 0:
        print("Removing duplicate zero channel from KKFLS.")
        data_kkfls = np.delete(data_kkfls, kkfls_zero_idx, axis=1)
        dist_kkfls = np.delete(dist_kkfls, kkfls_zero_idx)

    combined_data = np.concatenate((data_kkfls, data_terra), axis=1)
    combined_dist = np.concatenate((dist_kkfls, dist_terra))
   
    sort_indices = np.argsort(combined_dist)
    sorted_data = combined_data[:, sort_indices]
    sorted_dist = combined_dist[sort_indices]
    print("Combined patch created.")

    return sorted_data, times, sorted_dist



base_dir = "/Users/ed/Downloads"
kkfls_file_path = os.path.join(base_dir, "2023-08-03KKFLS.h5")
terra_file_path = os.path.join(base_dir, "2023-08-03TERRA.h5")

output_dir = os.path.join(base_dir, "das_plots_combined")
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "uberfigure.pdf")

try:
    kkfls_spool = dc.spool(kkfls_file_path)
    terra_spool = dc.spool(terra_file_path)
    kkfls_patch = kkfls_spool[0]
    terra_patch = terra_spool[0]
    print("Files loaded successfully.")

    kkfls_subsets = {
        "Dist_2k_3k": {
            'dist': (2000, 3000), # kkfls distance is negative in combine_patches.
            'P-Arrival': (10, 14), # start & end times in seconds
            'S-Arrival': (24, 28)
        },
        "Dist_30k_31k": {
            'dist': (30000, 31000),
            'P-Arrival': (8, 12),
            'S-Arrival': (22, 26)
        },
        "Dist_34k_35k": {
            'dist': (34000, 35000),
            'P-Arrival': (8, 12),
            'S-Arrival': (22, 26)
        },
        "Dist_45k_46k": {
            'dist': (45000, 46000),
            'P-Arrival': (8, 12),
            'S-Arrival': (21, 25)
        }
    }

    terra_subsets = {
        "Dist_8k_9k": {
            'dist': (8000, 9000),
            'P-Arrival': (10, 14),
            'S-Arrival': (23, 27)
        },
        "Dist_22k_23k": {
            'dist': (22000, 23000),
            'P-Arrival': (9, 13),
            'S-Arrival': (22, 26)
        },
        "Dist_30k_31k": {
            'dist': (30000, 31000),
            'P-Arrival': (9, 13),
            'S-Arrival': (21, 25)
        },
        "Dist_42k_43k": {
            'dist': (42000, 43000),
            'P-Arrival': (7, 11),
            'S-Arrival': (20, 24)
        },
        "Dist_53k_54k": {
            'dist': (53000, 54000),
            'P-Arrival': (6, 10),
            'S-Arrival': (19, 23)
        }
    }
    

    ref_plot_height_inches = 40
    zoom_plot_height_inches = 30 
    
    num_kkfls_zooms = sum(len(v) - 1 for v in kkfls_subsets.values())
    num_terra_zooms = sum(len(v) - 1 for v in terra_subsets.values()) 
    num_total_plots = 1 + num_terra_zooms + num_kkfls_zooms
    total_zoom_height = (num_terra_zooms + num_kkfls_zooms) * zoom_plot_height_inches
    total_fig_height = (ref_plot_height_inches * 1) + total_zoom_height
    height_ratios = [ref_plot_height_inches] + [zoom_plot_height_inches] * (num_terra_zooms + num_kkfls_zooms)
    
    fig, axes = plt.subplots(num_total_plots, 1, figsize=(20, total_fig_height), 
                             gridspec_kw={'height_ratios': height_ratios})

    if num_total_plots == 1:
        axes = [axes]
    
    try:
        ax_ref_combined = axes[0]
        
        t_start_kkfls = kkfls_patch.coords.min('time')
        t_start_terra = terra_patch.coords.min('time')
        
        decimated_kkfls = kkfls_patch.decimate(distance=4, filter_type='iir')
        processed_kkfls = decimated_kkfls.pass_filter(time=(1 * Hz, None))
        decimated_terra = terra_patch.decimate(distance=4, filter_type='iir')
        processed_terra = decimated_terra.pass_filter(time=(1 * Hz, None))
        ref_data, ref_times, ref_dists = combine_patches(processed_kkfls, processed_terra)

        matplotlib_wiggle(ax_ref_combined, ref_data, ref_times, scale=5, linewidth=0.5, alpha=0.3)
        ax_ref_combined.set_title("Combined Reference Plot: KKFLS (Negative) & TERRA (Positive)", fontsize=26)
        set_meter_ticks(ax_ref_combined, ref_dists, 2000)
        ax_ref_combined.invert_yaxis()

        # Commented-out section for drawing vertical gridlines
        # for fine-tuning reference zoom boxes
        # n_samples = len(ref_times)
        # sample_indices = np.arange(0, n_samples, 25)
        # times_to_plot = ref_times[sample_indices]
        # for t in times_to_plot:
        #     ax_ref_combined.axvline(t, color='lime', linestyle='-', linewidth=0.5, alpha=1.0)
        # print("1-s gridlines drawn.")
        
        current_plot_index = 1
        box_counter = 1 
        
        print("Generating TERRA zoom plots...")
        for subset_name, subset_info in terra_subsets.items():
            dist_range = subset_info['dist']
            
            for event_name, (start_sec, end_sec) in subset_info.items():
                if event_name == 'dist':
                    continue 

                ax_zoom = axes[current_plot_index]
                time_start = t_start_terra + np.timedelta64(start_sec, 's')
                time_end = t_start_terra + np.timedelta64(end_sec, 's')
                time_range = (time_start, time_end)

                zoomed_patch = terra_patch.select(time=time_range, distance=dist_range)                
                title = (f"Zoom {box_counter} (TERRA): {event_name} - {subset_name}\n"
                         f"Distance: {dist_range[0]}m - {dist_range[1]}m | "
                         f"Time: {start_sec}s to {end_sec}s")
                
                plot_zoomed_wiggle(zoomed_patch, ax_zoom, title)
                draw_zoom_box(ax_ref_combined, ref_dists, time_range, dist_range, str(box_counter))
                
                current_plot_index += 1
                box_counter += 1
        
        print("Generating KKFLS zoom plots...")
        for subset_name, subset_info in kkfls_subsets.items():
            dist_range = subset_info['dist']
            
            for event_name, (start_sec, end_sec) in subset_info.items():
                if event_name == 'dist':
                    continue 
                
                ax_zoom = axes[current_plot_index]
                time_start = t_start_kkfls + np.timedelta64(start_sec, 's')
                time_end = t_start_kkfls + np.timedelta64(end_sec, 's')
                time_range = (time_start, time_end)
                
                zoomed_patch = kkfls_patch.select(time=time_range, distance=dist_range)
                title = (f"Zoom {box_counter} (KKFLS): {event_name} - {subset_name}\n"
                         f"Distance: {dist_range[0]}m - {dist_range[1]}m | "
                         f"Time: {start_sec}s to {end_sec}s")
                
                plot_zoomed_wiggle(zoomed_patch, ax_zoom, title)
                kkfls_dist_range_neg = (dist_range[0] * -1, dist_range[1] * -1) 
                draw_zoom_box(ax_ref_combined, ref_dists, time_range, kkfls_dist_range_neg, str(box_counter))
                
                current_plot_index += 1
                box_counter += 1

        print(f"\nAll graphs plotted.")
        plt.savefig(output_file, bbox_inches='tight')
        plt.close(fig)
        print(f"\nSuccessfully saved to {output_file}")

    except Exception as e:
        print(f"Failed to create the master plot: {e} ---")
        if 'fig' in locals():
            plt.close(fig)

except Exception as e:
    print(f"An error occurred way early: {e}")

In [2]:
import dascore as dc
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as patches
from dascore.units import Hz
import numpy as np
import os

def matplotlib_wiggle(ax, data, times, dist_coords, scale, linewidth, alpha, highlight_dist=None):
    """
    Plots a DAS patch as a wiggle plot, optionally highlighting one trace in red.
    """
    if data.size == 0: 
        return

    n_samples, n_channels = data.shape
    
    # Identify the index to highlight red
    highlight_idx = None
    if highlight_dist is not None:
        highlight_idx = np.argmin(np.abs(dist_coords - highlight_dist))

    global_max = np.nanpercentile(np.abs(data), 99)
    if global_max <= 0 or not np.isfinite(global_max): global_max = 1.0
    
    for i in range(n_channels):
        trace = data[:, i]
        if np.all(np.isnan(trace)): continue
                
        scaled_trace = (np.nan_to_num(trace) / global_max) * scale
        
        # Determine color and style
        if i == highlight_idx:
            color, lw, l_alpha = ('red', linewidth * 1.5, 1.0)
        else:
            color, lw, l_alpha = ('black', linewidth, alpha)

        ax.plot(times, i + scaled_trace, color=color, linewidth=lw, alpha=l_alpha)
    
    ax.set_ylim(-1, n_channels)

def plot_zoomed_wiggle(patch, ax, title, highlight_dist=17500):
    """
    Processes and plots an UNDECIMATED DAS patch with a red highlight at 17,500m.
    """
    try:
        process1_patch = patch.dropna(dim="time", how="all")
        processed_patch = process1_patch.pass_filter(time=(1 * Hz, None))
        data = processed_patch.data
        times = processed_patch.coords.get_array('time')
        dists = processed_patch.coords.get_array('distance')
        
        # Pass distance array and highlight target to the wiggle function
        matplotlib_wiggle(ax, data, times, dists, scale=2, linewidth=3, alpha=0.5, highlight_dist=highlight_dist)         
        ax.set_title(title, fontsize=10)
        ax.invert_yaxis()
        
    except Exception as e:
        print(f"Failed to plot wiggle for {title}: {e}")
        ax.set_title(f"Failed to plot: {title}")

def set_meter_ticks(ax, dist_coords, tick_interval_m=1000):
    try:
        min_dist = np.min(dist_coords)
        max_dist = np.max(dist_coords)
        start_m = np.ceil(min_dist / tick_interval_m) * tick_interval_m
        end_m = np.floor(max_dist / tick_interval_m) * tick_interval_m
        meter_labels = np.arange(start_m, end_m + tick_interval_m, tick_interval_m)
        tick_indices = [np.argmin(np.abs(dist_coords - m)) for m in meter_labels]
        ax.set_yticks(tick_indices)
        ax.set_yticklabels([f"{int(m)}" for m in meter_labels])
        ax.set_ylabel("Distance (m)", fontsize=12)
    except Exception as e:
        ax.set_ylabel("Channel Index") 

# --- Directory and Path Setup ---
try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

data_dir = os.path.normpath(os.path.join(script_dir, "das_records", "good-events-3.2-up"))
output_dir = os.path.normpath(os.path.join(script_dir, "das_figures"))
os.makedirs(output_dir, exist_ok=True)

terra_file_path = os.path.join(data_dir, "ak0239vxdtm6_TERRA.h5")
output_file = os.path.join(output_dir, "uberfigure_terra_highlighted.pdf")

if not os.path.exists(terra_file_path):
    print(f"ERROR: File not found at {os.path.abspath(terra_file_path)}")
else:
    try:
        terra_spool = dc.spool(terra_file_path)
        terra_patch = terra_spool[0]
        
        terra_subsets = {
            "Dist_17.5k_Center": {'dist': (17400, 17600), 'P-Arrival': (9, 14)}
        }

        ref_plot_height_inches = 40
        zoom_plot_height_inches = 40 
        num_terra_zooms = sum(len(v) - 1 for v in terra_subsets.values()) 
        num_total_plots = 1 + num_terra_zooms
        height_ratios = [ref_plot_height_inches] + [zoom_plot_height_inches] * num_terra_zooms
        
        fig, axes = plt.subplots(num_total_plots, 1, figsize=(20, sum(height_ratios)), 
                                 gridspec_kw={'height_ratios': height_ratios})

        if num_total_plots == 1: axes = [axes]
        
        ax_ref = axes[0]
        t_start = terra_patch.coords.min('time')
        
        # 1. Reference Plot
        processed_ref = (terra_patch.decimate(distance=4).pass_filter(time=(1 * Hz, None)))
        ref_dists = processed_ref.coords.get_array('distance')
        
        matplotlib_wiggle(ax_ref, processed_ref.data, processed_ref.coords.get_array('time'), 
                          ref_dists, scale=5, linewidth=0.5, alpha=0.3)
        
        ax_ref.set_title(f"TERRA Reference Plot: {os.path.basename(terra_file_path)}", fontsize=26)
        set_meter_ticks(ax_ref, ref_dists, 2000)
        ax_ref.invert_yaxis()

        # 2. Zoom Plotting with Box logic
        current_plot_index = 1
        for subset_name, subset_info in terra_subsets.items():
            dist_range = subset_info['dist']
            for event_name, (start_sec, end_sec) in subset_info.items():
                if event_name == 'dist': continue 
                
                t_box_start = t_start + np.timedelta64(start_sec, 's')
                t_box_end = t_start + np.timedelta64(end_sec, 's')
                
                # Draw the Red Box on the Reference Plot
                y_low = np.argmin(np.abs(ref_dists - dist_range[0]))
                y_high = np.argmin(np.abs(ref_dists - dist_range[1]))
                rect = patches.Rectangle((t_box_start, y_low), t_box_end - t_box_start, y_high - y_low,
                                         linewidth=4, edgecolor='red', facecolor='none', linestyle='--', zorder=10)
                ax_ref.add_patch(rect)

                # Generate the Zoom Plot
                ax_zoom = axes[current_plot_index]
                zoomed_patch = terra_patch.select(time=(t_box_start, t_box_end), distance=dist_range)
                
                title = (f"Zoom (TERRA): {event_name}\n"
                         f"Distance: {dist_range[0]}m - {dist_range[1]}m | Time: {start_sec}s to {end_sec}s")
                plot_zoomed_wiggle(zoomed_patch, ax_zoom, title, highlight_dist=17500)
                
                current_plot_index += 1

        plt.savefig(output_file, bbox_inches='tight')
        plt.close(fig)
        print(f"Successfully saved to {output_file}")

    except Exception as e:
        print(f"Error: {e}")

ERROR: File not found at /Users/ed/research_code/das/minor_das_scripts/das_records/good-events-3.2-up/ak0239vxdtm6_TERRA.h5
